In [1]:
import gcsfs
import pandas as pd

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

In [2]:
EXISTING_GCS = "gs://calitp-analytics-data/data-analyses/ntd/"
existing_annual = pd.read_parquet(
    f"{EXISTING_GCS}annual_ridership_report_data.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [3]:
def extra_annual_rtpa_splitting(row):
    """
    Replace LA County Public Works agencies with their own RTPA
    For SCAG, use rtpa_name_split that mirrors each county.
    """
    # previously, used list to tag, but one NTD ID: 90271 was missing
    # use string to tag instead for resiliency
    # Los Angeles County - Department of Public Works, Transit Operations, East Los Angeles MB and DR
    # this was previously part of LACMTA, so now counts willl differ
    lacdpw_list = [
        "90269", "90270", "90272", "90273", "90274",
        "90275", "90276", "90277", "90278", "90279",
    ]

    # use 2 conditions to tag, since string can show with LACDPW before hyphen
    if (
        ("Los Angeles County - Department of Public Works" in row.source_agency) or 
        ("LACDPW" in row.source_agency)
    ):
        return "Los Angeles County Department of Public Works"
    elif row.rtpa_name == "Southern California Association of Governments":
        return row.rtpa_name_split
    else:
        return row.rtpa_name


In [4]:
def merge_new_df_with_crosswalk(
    filename: str = "annual"
):
    """
    """
    crosswalk = pd.read_parquet(
        f"{GCS_FILE_PATH}crosswalk2.parquet", 
        filesystem=gcsfs.GCSFileSystem(),
        columns = ["ntd_id_2022", "rtpa_name", "rtpa_name_split"]
    ).rename(columns = {"ntd_id_2022": "ntd_id"})

    df = pd.read_parquet(
        f"{GCS_FILE_PATH}{filename}.parquet",
        filesystem=gcsfs.GCSFileSystem(),
    ).merge(
        crosswalk,
        on = "ntd_id",
        how = "left"
    )
    
    # for annual, use rtpa_name_split
    df = df.assign(
        rtpa_name = df.apply(extra_annual_rtpa_splitting, axis=1)
    )

    return df

In [5]:
df = merge_new_df_with_crosswalk("annual")

In [6]:
df[df.ntd_id=="90271"].source_agency.value_counts()

source_agency
Los Angeles County - Department of Public Works, Transit Operations, East Los Angeles MB and DR    14
Name: count, dtype: int64

In [7]:
def counts_by_rtpa(
    df: pd.DataFrame,
    group_cols: list
) -> pd.DataFrame:
    """
    Use this to read in existing df vs new annual/monthly df
    and do groupby by rtpa_name or rtpa_name_split,
    and see how counts look overall.
    """
    df2 = (
        df
        .groupby(group_cols, dropna=False)
        .agg(
            total_upt=("upt", "sum"),
            n_agencies=("source_agency", "nunique"),
            agencies=pd.NamedAgg(column="source_agency", aggfunc=lambda x: list(set(x))),
            ntd_ids=pd.NamedAgg(column="ntd_id", aggfunc=lambda x: list(set(x))),
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by=group_cols + ["total_upt"], ascending=False)
        .reset_index(drop=True)
    )

    return df2

In [8]:
df2 = counts_by_rtpa(df.rename(columns = {"unlinked_passenger_trips": "upt"}), ["rtpa_name"])

In [9]:
existing_df2 = counts_by_rtpa(existing_annual, ["rtpa_name"])

In [10]:
compare_df = pd.merge(
    existing_df2, 
    df2,
    on = "rtpa_name",
    how = "outer",
    indicator=True
).astype({
    c: "Int64" 
    for c in ["total_upt_x", "total_upt_y", "n_agencies_x", "n_agencies_y"]}
)

In [11]:
compare_df.dtypes

rtpa_name         object
total_upt_x        Int64
n_agencies_x       Int64
agencies_x        object
ntd_ids_x         object
total_upt_y        Int64
n_agencies_y       Int64
agencies_y        object
ntd_ids_y         object
_merge          category
dtype: object

In [12]:
compare_df._merge.value_counts()

_merge
both          30
right_only     1
left_only      0
Name: count, dtype: int64

In [13]:
compare_df = compare_df.assign(
    missing_agency = compare_df.apply(
        lambda x:
        list(set([c for c in x.agencies_x if c not in x.agencies_y])) if x._merge=="both"
        else [], 
        axis=1),
    missing_ids = compare_df.apply(
        lambda x:
        list(set([c for c in x.ntd_ids_x if c not in x.ntd_ids_y])) if x._merge=="both"
        else [], 
        axis=1)
)

In [14]:
results = compare_df[(compare_df._merge=="both") & 
    (compare_df.total_upt_x != compare_df.total_upt_y)].reset_index(drop=True)

In [15]:
# this swaps where that one NTD ID that should have been in LACDPW was miscategorized in LACMTA
results

,rtpa_name,total_upt_x,n_agencies_x,agencies_x,ntd_ids_x,total_upt_y,n_agencies_y,agencies_y,ntd_ids_y,_merge,missing_agency,missing_ids
0,Los Angeles County Department of Public Works,3825872,10,[Los Angeles County - Department of Public Wor...,"[90269, 90278, 90277, 90276, 90270, 90279, 902...",7511111,11,[Los Angeles County - Department of Public Wor...,"[90269, 90278, 90277, 90276, 90270, 90271, 902...",both,[],[]
1,Los Angeles County Metropolitan Transportation...,2712173584,63,"[City of Avalon, City of Calabasas (COC) - Pub...","[90281, 90282, 90262, 90259, 90121, 90284, 900...",2708488345,62,"[City of Avalon, City of Calabasas (COC) - Pub...","[90281, 90282, 90262, 90259, 90121, 90284, 900...",both,[Los Angeles County - Department of Public Wor...,[90271]


In [16]:
results.missing_agency.iloc[1]

['Los Angeles County - Department of Public Works, Transit Operations, East Los Angeles MB and DR']

In [18]:
for rtpa in results.rtpa_name.unique():
    print(rtpa)
    subset_df = results[results.rtpa_name==rtpa].reset_index(drop=True)
    print(subset_df.missing_agency.iloc[0])
    print(subset_df.missing_ids.iloc[0])
    print("******************************************************************")

Los Angeles County Department of Public Works
[]
[]
******************************************************************
Los Angeles County Metropolitan Transportation Authority
['Los Angeles County - Department of Public Works, Transit Operations, East Los Angeles MB and DR']
['90271']
******************************************************************
